# 00 · Optimizer testleri

**Amaç:** `optim/optimizers.py` içindeki makaleye sadık `PaperAdam` ve `PaperAMSGrad` implementasyonunun doğruluğunu göstermek.

1. Birim testleri (`tests/test_optimizers.py`)
   - bias correction + ε açıkken `torch.optim.Adam` (ve `amsgrad=True`) ile birebir aynı sonuç
   - AMSGrad'da v̂ hiç azalmıyor
   - fonksiyonel `paper_adam_step` = `Optimizer` sınıfı
   - vektörize (broadcast) konfigürasyonlar = tek tek koşular
   - ε = 0 iken sıfır gradyan NaN üretmiyor
   - sentetik problemde Adam +1'e, AMSGrad −1'e doğru gidiyor
2. GPU'da (float32) `torch.optim.Adam` ile eşleşme kontrolü
3. Makalenin sentetik problemi (Şekil 1, online) için hızlı bir görsel kontrol

**Kaggle ayarları:** Internet **On**. Accelerator: GPU (2. bölüm için; diğerleri CPU'da da çalışır).

## Makaleden sapmalar
Bu notebook bir doğrulama notebook'u, makalenin bir deneyini reproduce etmiyor. Bölüm 3'teki görsel kontrolde:
- α_t = 0.5/√t (makale sentetik deneydeki α'yı vermiyor)
- 200k iterasyon (makale ~7M koşuyor; tam versiyon `01_synthetic_online` notebook'unda)
- bias correction yok, ε = 0, projeksiyon [−1, 1] (makalenin analizindeki gibi)

In [ ]:
# --- Kurulum: repo'yu çek, yolları ayarla ---
import os, sys, subprocess
REPO = "https://github.com/ybilgici/DeepLearning_Proje.git"
if os.path.exists("/kaggle"):
    ROOT = "/kaggle/working/DeepLearning_Proje"
    if not os.path.exists(ROOT):
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=True)
else:  # lokal çalıştırma: notebooks/ klasörünün bir üstü repo kökü
    ROOT = os.path.abspath("..")
sys.path.insert(0, ROOT)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| torch", torch.__version__)
print("Commit:", subprocess.run(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())

## 1. Birim testleri

In [ ]:
import importlib.util
if importlib.util.find_spec("pytest") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", os.path.join(ROOT, "tests")],
                   capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "Testler başarısız!"

## 2. GPU (float32) üzerinde `torch.optim.Adam` ile eşleşme

In [ ]:
from optim import PaperAdam, PaperAMSGrad, paper_adam_step, init_state

def gpu_match(amsgrad, steps=200):
    torch.manual_seed(0)
    X = torch.randn(1024, 20, device=DEVICE); y = torch.randn(1024, 1, device=DEVICE)
    def make():
        torch.manual_seed(1)
        return torch.nn.Sequential(torch.nn.Linear(20, 64), torch.nn.ReLU(), torch.nn.Linear(64, 1)).to(DEVICE)
    m1, m2 = make(), make()
    cls = PaperAMSGrad if amsgrad else PaperAdam
    o1 = cls(m1.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8, bias_correction=True)
    o2 = torch.optim.Adam(m2.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8, amsgrad=amsgrad)
    for i in range(steps):
        idx = torch.randint(0, 1024, (128,), device=DEVICE, generator=None)
        for m, o in ((m1, o1), (m2, o2)):
            o.zero_grad(); torch.nn.functional.mse_loss(m(X[idx]), y[idx]).backward(); o.step()
    p1 = torch.cat([p.flatten() for p in m1.parameters()]); p2 = torch.cat([p.flatten() for p in m2.parameters()])
    return (p1 - p2).abs().max().item()

for ams in (False, True):
    d = gpu_match(ams)
    print(f"amsgrad={ams}: max |fark| = {d:.2e}")
    assert d < 1e-4

## 3. Sentetik problem: hızlı görsel kontrol

f_t(x) = 1010x (t mod 101 = 1), aksi halde −10x, F = [−1, 1]. Optimum x = −1.
Beklenti: Adam +1'e yapışır, AMSGrad −1'e iner. CPU'da ~1 dakika sürer.

In [ ]:
import matplotlib.pyplot as plt

T = 200_000
def run(amsgrad):
    x = torch.ones(1, dtype=torch.float64); st = init_state(x); xs = []
    big, small = torch.tensor([1010.0], dtype=torch.float64), torch.tensor([-10.0], dtype=torch.float64)
    for t in range(1, T + 1):
        g = big if t % 101 == 1 else small
        x = paper_adam_step(x, g, st, lr=0.5, beta1=0.9, beta2=0.99, lr_schedule="sqrt",
                            clamp=(-1, 1), amsgrad=amsgrad)
        if t % 100 == 0: xs.append(x.item())
    return xs

xa, xm = run(False), run(True)
steps = [100 * (i + 1) for i in range(len(xa))]
plt.figure(figsize=(7, 3.2))
plt.plot(steps, xa, "--", color="#d1495b", label="Adam")
plt.plot(steps, xm, "-", color="#2e86ab", label="AMSGrad")
plt.xlabel("iterasyon"); plt.ylabel("$x_t$"); plt.ylim(-1.1, 1.1); plt.grid(alpha=.3); plt.legend()
plt.title("Sentetik problem (online): Adam vs AMSGrad"); plt.tight_layout(); plt.show()
print(f"Son x — Adam: {xa[-1]:+.4f}   AMSGrad: {xm[-1]:+.4f}")
assert xa[-1] > 0.95 and xm[-1] < -0.9